# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SalehAl-Nassar/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.**
Unit: one content page, scored by how likely it needs a refresh. Proxy label = did its GSC position get worse from H1 to H2.

## 1. Unit of analysis + time window

**Unit:** one page (`content_hash_id`), rolled up to the page-month level.
**Feature window (H1):** March 1-15. Everything a feature might use comes from here.
**Label window (H2):** March 16-31. The proxy label lives here, so it's never seen during feature building.

Why split the month like this? Pretty straightforward — if your feature and label come from the same dates, you're just measuring the present, not predicting anything. The starter CSV does this (trend_direction is computed from the same 90-day window), but here we keep them separate.

March 2026: 9.8M daily rows, ~331K pages, 55 clients. June 2026 (`_sample` table) is sealed for final eval, not for developing the label.

In [26]:
!pip install -q duckdb pandas pyarrow
import os, duckdb
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

# -- token ---------------------------------------------------------
token = os.environ.get('HF_TOKEN')
if token is None:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except (ImportError, Exception):
        pass
if token is None:
    from pathlib import Path
    for p in [Path('../../.env'), Path('.env'), Path.home() / '.env']:
        if p.exists():
            with open(p) as f:
                for line in f:
                    if line.strip().startswith('hf_token='):
                        token = line.strip().split('=', 1)[1].strip()
                        break
            break
if token is None:
    raise RuntimeError('Set HF_TOKEN as a Colab secret (key icon in sidebar) or in .env')

# -- download via huggingface_hub (proven auth) --------------------
from huggingface_hub import hf_hub_download
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '600'

dim_content = hf_hub_download('FlyRank/internship-warehouse', 'dim_content.parquet',
                              repo_type='dataset', token=token)
fact_mar = hf_hub_download('FlyRank/internship-warehouse',
                           'fact_content_daily_performance/month=2026-03/data_0.parquet',
                           repo_type='dataset', token=token)

# -- DuckDB on local files -----------------------------------------
con = duckdb.connect()

# 1a. March partition overview
r = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(DISTINCT content_hash_id) AS distinct_pages,
           COUNT(DISTINCT client_hash_id) AS distinct_clients,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM read_parquet('{fact_mar}')
""").to_df()
print('March 2026 partition - overview')
print(r.to_string(index=False))
print()

# 1b. H1 / H2 split
r2 = con.sql(f"""
    SELECT CASE WHEN EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15 THEN 'H1 (1-15)'
                WHEN EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31 THEN 'H2 (16-31)'
                ELSE 'other' END AS half,
           COUNT(*) AS rows,
           COUNT(DISTINCT content_hash_id) AS pages,
           MIN(report_date) AS min_dt,
           MAX(report_date) AS max_dt
    FROM read_parquet('{fact_mar}')
    GROUP BY half ORDER BY half
""").to_df()
print('H1 / H2 split')
print(r2.to_string(index=False))
print()

# 1c. Cache page-month aggregate
print('Caching page-month aggregate to work/outputs/mar_page_month.parquet ...')
con.sql(f"""
    CREATE OR REPLACE TABLE h1 AS
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_h1,
               SUM(gsc_clicks) AS clicks_h1,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h1,
               SUM(ga4_sessions) AS sessions_h1,
               SUM(ga4_engaged_sessions) AS engaged_sessions_h1,
               SUM(sessions_organic) AS sessions_organic_h1
        FROM read_parquet('{fact_mar}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
        GROUP BY content_hash_id, client_hash_id;
    CREATE OR REPLACE TABLE h2 AS
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_h2,
               SUM(gsc_clicks) AS clicks_h2,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_h2,
               SUM(ga4_sessions) AS sessions_h2,
               SUM(ga4_engaged_sessions) AS engaged_sessions_h2,
               SUM(sessions_organic) AS sessions_organic_h2
        FROM read_parquet('{fact_mar}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31
        GROUP BY content_hash_id;
    CREATE OR REPLACE TABLE page_month AS
        SELECT h1.content_hash_id, h1.client_hash_id,
               h1.impressions_h1, h1.clicks_h1, h1.avg_position_h1,
               h1.sessions_h1, h1.engaged_sessions_h1, h1.sessions_organic_h1,
               h2.impressions_h2, h2.clicks_h2, h2.avg_position_h2,
               h2.sessions_h2, h2.engaged_sessions_h2, h2.sessions_organic_h2,
               dc.word_count, dc.search_volume, dc.competition_level,
               dc.main_intent, dc.content_type, dc.content_created_date,
               dc.last_optimized_date,
               DATEDIFF('day', COALESCE(dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE) AS content_age_days,
               GREATEST(0, DATEDIFF('day', COALESCE(dc.last_optimized_date, dc.content_created_date, '2026-03-01'::DATE), '2026-03-01'::DATE)) AS days_since_update
        FROM h1
        INNER JOIN h2 ON h1.content_hash_id = h2.content_hash_id
        LEFT JOIN read_parquet('{dim_content}') dc
               ON h1.content_hash_id = dc.content_hash_id
""")
os.makedirs('work/outputs', exist_ok=True)
con.sql("COPY page_month TO 'work/outputs/mar_page_month.parquet' (FORMAT PARQUET)")

r3 = con.sql('SELECT COUNT(*) AS n FROM page_month').to_df()
print(f'Cached: {r3.n[0]} page-month rows')
con.sql('SELECT * FROM page_month LIMIT 3').to_df()

March 2026 partition - overview
 total_rows  distinct_pages  distinct_clients first_date  last_date
    9841378          331437                55 2026-03-01 2026-03-31

H1 / H2 split
      half    rows  pages     min_dt     max_dt
 H1 (1-15) 4642255 319759 2026-03-01 2026-03-15
H2 (16-31) 5199123 331436 2026-03-16 2026-03-31

Caching page-month aggregate to work/outputs/mar_page_month.parquet ...
Cached: 319758 page-month rows


,content_hash_id,client_hash_id,impressions_h1,clicks_h1,avg_position_h1,sessions_h1,engaged_sessions_h1,sessions_organic_h1,impressions_h2,clicks_h2,...,sessions_organic_h2,word_count,search_volume,competition_level,main_intent,content_type,content_created_date,last_optimized_date,content_age_days,days_since_update
0,content_149355c8dfc3f8e1,client_2094c6eb080311d5,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,4038,0,LOW,informational,keyword article,2026-01-22,NaT,38,38
1,content_14a3d47ccd0d15dc,client_2094c6eb080311d5,2.0,0.0,5.5,0.0,0.0,0.0,0.0,0.0,...,0.0,3864,0,LOW,informational,keyword article,2025-12-09,NaT,82,82
2,content_14a6f92117604fef,client_2094c6eb080311d5,0.0,0.0,NaN,0.0,0.0,0.0,90.0,0.0,...,0.0,2867,0,LOW,informational,keyword article,2025-12-11,2026-05-20,80,0


## 2. Fields: feature / label / context / excluded

### Features (knowable before H2 opens)

| Field | Source | Description |
|---|---|---|
| `impressions_h1` | fact (days 1-15) | Total GSC impressions in the feature window |
| `clicks_h1` | fact (days 1-15) | Total GSC clicks |
| `avg_position_h1` | fact (days 1-15) | Average GSC position (lower = better). ~52% NULL for pages with zero impressions in H1 |
| `sessions_h1` | fact (days 1-15) | GA4 sessions. ~31% NULL where `ga4_data_available = FALSE` |
| `engaged_sessions_h1` | fact (days 1-15) | GA4 engaged sessions |
| `sessions_organic_h1` | fact (days 1-15) | Organic search sessions |
| `word_count` | dim_content | Content length in words. ~34% NULL (pages without dim_content match) |
| `search_volume` | dim_content | Keyword-level search demand. ~18% NULL |
| `competition_level` | dim_content | Keyword competition (LOW / MEDIUM / HIGH). ~18% NULL |
| `main_intent` | dim_content | Keyword intent (informational, commercial, transactional, navigational). ~18% NULL |
| `content_type` | dim_content | Type of content page. 0% NULL |
| `content_age_days` | derived | Days since `content_created_date` to March 1, 2026. 0% NULL |
| `days_since_update` | derived | Days since `last_optimized_date` (or created date) to March 1. Capped at 0. |

### Label / Proxy (from H2, days 16-31)

| Field | Description |
|---|---|
| `impressions_h2`, `clicks_h2`, `avg_position_h2` | H2 metrics used to compute the proxy |
| `proxy_decline` (derived) | **1** if avg_position worsened >=10% from H1 to H2, **0** otherwise. Favors recall over precision. Observed rate: ~44% of pages with non-NULL position in both halves. |

### Context (join / group / split only — never features)

`content_hash_id`, `client_hash_id` — just IDs for joining and grouping, never fed to a model.

### Excluded

| Field | Why excluded |
|---|---|
| `report_date` | Partition column; each row is already page-month |
| `month` | Redundant partition label |
| `keyword_hash_id`, `url_hash_id` | Raw identifiers, not signals |
| `provider_used`, `model_used` | Product implementation detail; not a content signal |
| `is_published`, `is_deleted` | Product status flags, not performance signals |
| `ga4_data_available`, `gsc_data_available` | Gating flags, not features — but they *are* checked to understand missingness patterns (see Section 4) |

In [27]:
# -- 2a. reload cached page-month
con2 = duckdb.connect()
con2.execute("CREATE OR REPLACE VIEW pm AS SELECT * FROM read_parquet('work/outputs/mar_page_month.parquet')")

# -- 2b. columns
print('Columns in page-month cache:')
print([c[0] for c in con2.sql('DESCRIBE pm').fetchall()])
print()

# -- 2c. missingness overview
r_miss = con2.sql("""
    SELECT COUNT(*) AS total_rows,
           AVG(CASE WHEN impressions_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_impressions_h1,
           AVG(CASE WHEN clicks_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_clicks_h1,
           AVG(CASE WHEN avg_position_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_avg_pos_h1,
           AVG(CASE WHEN sessions_h1 IS NULL THEN 1.0 ELSE 0 END) AS miss_sessions_h1,
           AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END) AS miss_word_count,
           AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END) AS miss_search_volume,
           AVG(CASE WHEN competition_level IS NULL THEN 1.0 ELSE 0 END) AS miss_competition,
           AVG(CASE WHEN main_intent IS NULL THEN 1.0 ELSE 0 END) AS miss_intent,
           AVG(CASE WHEN content_type IS NULL THEN 1.0 ELSE 0 END) AS miss_content_type,
           AVG(CASE WHEN content_age_days IS NULL THEN 1.0 ELSE 0 END) AS miss_age,
           AVG(CASE WHEN days_since_update IS NULL THEN 1.0 ELSE 0 END) AS miss_days_since_update
    FROM pm
""").to_df()
print('Missingness rates:')
for col in r_miss.columns:
    v = r_miss[col].values[0]
    if col != 'total_rows':
        print(f'  {col}: {v:.4f} ({v*100:.1f}%)')
print()

# -- 2d. sample
con2.sql('SELECT * FROM pm LIMIT 3').to_df()

Columns in page-month cache:
['content_hash_id', 'client_hash_id', 'impressions_h1', 'clicks_h1', 'avg_position_h1', 'sessions_h1', 'engaged_sessions_h1', 'sessions_organic_h1', 'impressions_h2', 'clicks_h2', 'avg_position_h2', 'sessions_h2', 'engaged_sessions_h2', 'sessions_organic_h2', 'word_count', 'search_volume', 'competition_level', 'main_intent', 'content_type', 'content_created_date', 'last_optimized_date', 'content_age_days', 'days_since_update']

Missingness rates:
  miss_impressions_h1: 0.0000 (0.0%)
  miss_clicks_h1: 0.0000 (0.0%)
  miss_avg_pos_h1: 0.5247 (52.5%)
  miss_sessions_h1: 0.3053 (30.5%)
  miss_word_count: 0.3357 (33.6%)
  miss_search_volume: 0.1825 (18.2%)
  miss_competition: 0.1847 (18.5%)
  miss_intent: 0.1780 (17.8%)
  miss_content_type: 0.0000 (0.0%)
  miss_age: 0.0000 (0.0%)
  miss_days_since_update: 0.0000 (0.0%)



,content_hash_id,client_hash_id,impressions_h1,clicks_h1,avg_position_h1,sessions_h1,engaged_sessions_h1,sessions_organic_h1,impressions_h2,clicks_h2,...,sessions_organic_h2,word_count,search_volume,competition_level,main_intent,content_type,content_created_date,last_optimized_date,content_age_days,days_since_update
0,content_149355c8dfc3f8e1,client_2094c6eb080311d5,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,4038,0,LOW,informational,keyword article,2026-01-22,NaT,38,38
1,content_14a3d47ccd0d15dc,client_2094c6eb080311d5,2.0,0.0,5.5,0.0,0.0,0.0,0.0,0.0,...,0.0,3864,0,LOW,informational,keyword article,2025-12-09,NaT,82,82
2,content_14a6f92117604fef,client_2094c6eb080311d5,0.0,0.0,NaN,0.0,0.0,0.0,90.0,0.0,...,0.0,2867,0,LOW,informational,keyword article,2025-12-11,2026-05-20,80,0


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets checked here. If there's no query for it, don't trust it.

In [28]:
# -- 3a. Grain check
print('=== Grain: rows with duplicate content_hash_id ===')
bad = con2.sql("""
    SELECT content_hash_id, COUNT(*) AS c
    FROM pm GROUP BY content_hash_id HAVING COUNT(*) > 1 LIMIT 5
""").to_df()
if len(bad) == 0:
    print('  PASS - no duplicates. 1 row = 1 page.')
else:
    print(f'  FAIL - {len(bad)} duplicates exist')
print()

# -- 3b. Pages per client
print('=== Pages per client (top 5) ===')
print(con2.sql("""
    SELECT client_hash_id, COUNT(*) AS pages
    FROM pm GROUP BY client_hash_id ORDER BY pages DESC LIMIT 5
""").to_df().to_string(index=False))
print()

# -- 3c. H1 / H2 stats
print('=== H1 stats (days 1-15) ===')
print(con2.sql("""
    SELECT COUNT(*) AS pages, MIN(impressions_h1) AS min_imp,
           AVG(impressions_h1) AS avg_imp, MAX(impressions_h1) AS max_imp,
           AVG(clicks_h1) AS avg_clicks, AVG(avg_position_h1) AS avg_pos
    FROM pm
""").to_df().to_string(index=False))
print()

print('=== H2 stats (days 16-31) ===')
print(con2.sql("""
    SELECT COUNT(*) AS pages, MIN(impressions_h2) AS min_imp,
           AVG(impressions_h2) AS avg_imp, MAX(impressions_h2) AS max_imp,
           AVG(clicks_h2) AS avg_clicks, AVG(avg_position_h2) AS avg_pos
    FROM pm
""").to_df().to_string(index=False))
print()

# -- 3d. dim_content missingness
print('=== Rows with NULL word_count ===')
r_miss_dc = con2.sql("""
    SELECT COUNT(*) AS total_nodc, COUNT(DISTINCT client_hash_id) AS clients_nodc
    FROM pm WHERE word_count IS NULL
""").to_df()
total_rows = r_miss.total_rows[0]
print(f'  Without dim_content: {r_miss_dc.total_nodc[0]} ({r_miss_dc.total_nodc[0]/total_rows*100:.1f}%)')
print(f'  Affected clients: {r_miss_dc.clients_nodc[0]}')
print()

# -- 3e. Proxy label distribution
print('=== Proxy label: avg_position worsened >=10% from H1 to H2 ===')
r_label = con2.sql("""
    SELECT CASE WHEN avg_position_h2 > avg_position_h1 * 1.10 THEN 1 ELSE 0 END AS proxy_decline,
           COUNT(*) AS pages
    FROM pm WHERE avg_position_h1 IS NOT NULL AND avg_position_h2 IS NOT NULL
    GROUP BY proxy_decline ORDER BY proxy_decline
""").to_df()
print(r_label.to_string(index=False))
if len(r_label) >= 2:
    d = r_label[r_label.proxy_decline == 1].pages.values[0]
    t = r_label.pages.sum()
    print(f'  Decline rate: {d}/{t} = {d/t*100:.1f}%')
print()

# -- 3f. Confirm no gsc_ctr column
print('=== Confirming: no gsc_ctr column ===')
cols = [c[0] for c in con2.sql(f"DESCRIBE SELECT * FROM read_parquet('{fact_mar}')").fetchall()]
has = 'gsc_ctr' in cols
if not has:
    print('  CONFIRMED: gsc_ctr does not exist. Compute as clicks / impressions.')
else:
    print('  UNEXPECTED: gsc_ctr exists!')

=== Grain: rows with duplicate content_hash_id ===
  PASS - no duplicates. 1 row = 1 page.

=== Pages per client (top 5) ===
         client_hash_id  pages
client_625b6439094e23e4  31887
client_3ffa76342f366962  31106
client_73cda7b4e4f265ea  28235
client_08a6a72ff48e62c0  27251
client_62f4a7e64f5e0096  24456

=== H1 stats (days 1-15) ===
 pages  min_imp    avg_imp  max_imp  avg_clicks   avg_pos
319758      0.0 398.766896 161575.0    1.203451 15.653035

=== H2 stats (days 16-31) ===
 pages  min_imp    avg_imp  max_imp  avg_clicks   avg_pos
319758      0.0 477.420803 455549.0    1.360892 16.352913

=== Rows with NULL word_count ===
  Without dim_content: 107348 (33.6%)
  Affected clients: 15

=== Proxy label: avg_position worsened >=10% from H1 to H2 ===
 proxy_decline  pages
             0  79677
             1  61790
  Decline rate: 61790/141467 = 43.7%

=== Confirming: no gsc_ctr column ===
  CONFIRMED: gsc_ctr does not exist. Compute as clicks / impressions.


## 4. Data limits

*Stuff this data just can't tell you, and why it matters.*

### 4a. GA4 is barely there
Only ~4% of March rows have `ga4_data_available = TRUE`. The rest are FALSE (65%) or just NULL (31%). So `sessions_h1` and friends will be missing for most pages. If your feature set depends on GA4, you're working with a tiny subset. Better to lean on GSC-only features and use a `has_ga4` flag.

### 4b. H1-H2 inner join drops almost nothing
Only 1 page out of 331K had H1 data but no H2 data. Survivorship bias is essentially zero here.

### 4c. Only 44% of pages get a proxy label
About 52.5% of pages have NULL `avg_position_h1` (zero impressions in H1). Since the label needs position in both halves, only ~141K of 320K pages get scored. Among those, ~44% are declining.

### 4d. The proxy is a heuristic, not truth
`proxy_decline = 1` just means the rank number got >=10% worse. That's not the same as "a human editor would refresh this page." Real content-refresh decisions need editorial judgment.

### 4e. One month only
Everything here is March 2026. June is sealed. What works in March might not hold in other months.

### 4f. No gsc_ctr column
Have to compute it yourself as `clicks / NULLIF(impressions, 0)`. Verified in the queries.

### 4g. Negative days_since_update exists
Some `last_optimized_date` values are after March 1 (future-dated). Capped at 0.

### 4h. Registration-day trap
Pages created after March 1 have zero history before their creation date. Their early H1 zeros aren't bad performance — they're absence. ~5.3% of pages have this issue.

### 4i. CTR convention differs from the starter CSV
In the starter CSV, `ctr = 0.76` means 0.76%. Our warehouse CTR is a raw ratio (0.0076). Both are fine, just pick one and stick with it.

In [29]:
# -- 4a. GA4 data availability (raw March 2026)
print('=== GA4 data availability (raw March 2026) ===')
r_ga4 = con.sql(f"""
    SELECT ga4_data_available, COUNT(*) AS rows,
           COUNT(DISTINCT content_hash_id) AS pages,
           COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{fact_mar}')
    GROUP BY ga4_data_available ORDER BY ga4_data_available
""").to_df()
print(r_ga4.to_string(index=False))
print()

# -- 4b. H1-only pages
print('=== Pages with H1 data but not H2 ===')
r_mar_pages = con.sql(f"""SELECT COUNT(DISTINCT content_hash_id) AS n FROM read_parquet('{fact_mar}')""").to_df()
r_h1_only = con.sql(f"""
    SELECT COUNT(*) AS h1_only FROM (
        SELECT content_hash_id FROM read_parquet('{fact_mar}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15 GROUP BY content_hash_id
        EXCEPT
        SELECT content_hash_id FROM read_parquet('{fact_mar}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 16 AND 31 GROUP BY content_hash_id
    )
""").to_df()
print(f'  Pages in March: {r_mar_pages.n[0]}')
print(f'  H1 only (no label): {r_h1_only.h1_only[0]}')
print()

# -- 4c. Pages with <3 days in H1
print('=== Pages with <3 days of data in H1 ===')
r_noisy = con.sql(f"""
    SELECT COUNT(*) AS n FROM (
        SELECT content_hash_id FROM read_parquet('{fact_mar}')
        WHERE EXTRACT(DAY FROM report_date) BETWEEN 1 AND 15
        GROUP BY content_hash_id HAVING COUNT(*) < 3
    )
""").to_df()
print(f'  <3 days: {r_noisy.n[0]} ({r_noisy.n[0]/r_mar_pages.n[0]*100:.1f}%)')
print()

# -- 4d. Registration-day trap
print('=== Pages created after March 1 ===')
r_late = con2.sql("""
    SELECT COUNT(*) AS n FROM read_parquet('work/outputs/mar_page_month.parquet')
    WHERE content_created_date > '2026-03-01'::DATE
""").to_df()
r_total = con2.sql("""SELECT COUNT(*) AS n FROM read_parquet('work/outputs/mar_page_month.parquet')""").to_df()
print(f'  Late-start: {r_late.n[0]} ({r_late.n[0]/r_total.n[0]*100:.1f}%)')
print()

# -- 4e. Client gap
print('=== Clients in fact vs dim_content ===')
r_fact = con.sql(f"""SELECT COUNT(DISTINCT client_hash_id) AS n FROM read_parquet('{fact_mar}')""").to_df()
r_dc = con.sql(f"""SELECT COUNT(DISTINCT client_hash_id) AS n FROM read_parquet('{dim_content}')""").to_df()
print(f'  Fact March: {r_fact.n[0]}  dim_content: {r_dc.n[0]}  gap: {r_dc.n[0] - r_fact.n[0]}')

=== GA4 data availability (raw March 2026) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 ga4_data_available    rows  pages  clients
              False 6408671 260497       43
               True  413966  90489       41
               <NA> 3018741 166412       22

=== Pages with H1 data but not H2 ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  Pages in March: 331437
  H1 only (no label): 1

=== Pages with <3 days of data in H1 ===
  <3 days: 1859 (0.6%)

=== Pages created after March 1 ===
  Late-start: 17066 (5.3%)

=== Clients in fact vs dim_content ===
  Fact March: 55  dim_content: 84  gap: 29


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.